<a href="https://colab.research.google.com/github/Alltingzrpozz/Alltingzrpozz/blob/main/Advanced_Spatial_Feature_Extraction_Convolutional_Neural_Network_(CNN)_via_Transfer_Learning(3)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#This notebook constructs an enterprise-grade image classification pipeline. By leveraging a pre-trained VGG16 base as a spatial feature extractor, we analyze 224x224 MRI tensors to diagnose four distinct stages of cognitive decline. This architecture circumvents the need for structured clinical data by relying entirely on deep spatial pattern recognition.

# 1.0 Enterprise Environment Setup & Dependency Injection

Initialization of the TensorFlow backend, secure mounting of the Google Drive environment, and definition of global file paths

In [ ]:
# ==============================================================================
# 1.0 ENVIRONMENT SETUP & PATH DEFINITION
# ==============================================================================
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input
from google.colab import drive

print("--- CONFIGURING SECURE ENVIRONMENT ---")

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the exact Base Path from your environment
BASE_DIR = '/content/drive/MyDrive/Study/MSc AI/data sets/Dementia - Image DL Analysis for 10 min vlog/Alzheimers_Detection_dataset/'

# 3. Define specific sub-directories securely
CSV_DIR = os.path.join(BASE_DIR, 'CSV_datafiles/')
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train/')
VALID_IMG_DIR = os.path.join(BASE_DIR, 'valid/')
TEST_IMG_DIR  = os.path.join(BASE_DIR, 'test/')

print(f"TensorFlow Version: {tf.__version__}")
print("Environment Initialized. Paths secured.")

# 2.0 Secure Image Data Pipeline & Spatial Synchronization


We engineer a custom `tf.data` pipeline to prevent memory overflow (I/O bottlenecks). This module ingests the Kaggle CSV mapping files and dynamically stitches the local file paths to the raw MRI spatial tensors for highly optimized, cached batch generation.

In [ ]:
# ==============================================================================
# 2.0 DATA INGESTION & PIPELINE CREATION
# ==============================================================================
print("\n--- BUILDING TF.DATA PIPELINE ---")

# 1. Load the mapping CSVs
train_df = pd.read_csv(os.path.join(CSV_DIR, '_train_classes.csv'))
valid_df = pd.read_csv(os.path.join(CSV_DIR, '_valid_classes.csv'))
test_df  = pd.read_csv(os.path.join(CSV_DIR, '_test_classes.csv'))

# 2. Path Synchronization: Stitch the full directory path to the filename
# Note: Ensure the column name in your CSV is exactly 'filename' or 'Path'
train_df['image_file_path'] = TRAIN_IMG_DIR + train_df['filename'] # Change 'filename' to 'Path' if needed
valid_df['image_file_path'] = VALID_IMG_DIR + valid_df['filename']
test_df['image_file_path']  = TEST_IMG_DIR + test_df['filename']

def load_and_preprocess_image(image_path, labels):
    """Ingests raw MRI images and applies architecture-specific VGG16 mathematical preprocessing."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = preprocess_input(img)
    return img, labels

def create_image_dataset(df, batch_size=32, is_training=True):
    """Transforms standard dataframes into pre-fetched, cached TensorFlow datasets."""
    image_paths = df['image_file_path'].values
    # Grab the one-hot encoded columns directly from the CSV
    labels = df[['ND', 'VMD', 'MD', 'MoD']].astype('float32').values

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

    # Cache in RAM to drastically speed up epoch training times
    dataset = dataset.cache()

    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# 3. Pipeline Execution
train_dataset = create_image_dataset(train_df, is_training=True)
val_dataset = create_image_dataset(valid_df, is_training=False)
test_dataset = create_image_dataset(test_df, is_training=False)

print("Spatial image pipeline successfully constructed and cached.")

In [ ]:
print("--- CALCULATING CLASS WEIGHTS FOR IMBALANCED DATASET ---")
from sklearn.utils import class_weight

# Extract one-hot encoded labels from train_df
# Assuming columns are in the order of class_labels: ND, VMD, MD, MoD
y_train_labels_one_hot = train_df[['ND', 'VMD', 'MD', 'MoD']].values

# Convert one-hot encoded labels to single integer labels for class_weight utility
# np.argmax will give the index of the '1' in the one-hot vector
y_train_labels = np.argmax(y_train_labels_one_hot, axis=1)

# Define class labels for clarity (must match the order of one-hot encoding)
class_labels_for_weights = ['Non-Demented', 'Very Mild', 'Mild', 'Moderate']

# Calculate class weights using 'balanced' mode
# This automatically adjusts weights inversely proportional to class frequencies
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)

# Convert the array to a dictionary for Keras fit method
class_weights_dict = dict(enumerate(class_weights_array))

print("Calculated Class Weights (dictionary format for Keras fit):")
for idx, weight in class_weights_dict.items():
    print(f"  Class {idx} ({class_labels_for_weights[idx]}): {weight:.2f}")

print("Class weights prepared.")

# 3.0 Spatial Architecture: VGG16 Base & Custom Diagnostic Head

Deploying Transfer Learning. We instantiate a frozen VGG16 network to act purely as a spatial kernel, flattening the cerebral textures into a 1D latent vector. A custom, high-dropout dense network interprets this representation to output the diagnostic stage.

In [ ]:
# ==============================================================================
# 3.0 ARCHITECTURE ENGINEERING
# ==============================================================================
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

print("--- COMPILING CNN TOPOLOGY ---")

image_input = Input(shape=(224, 224, 3), name="MRI_Scan_Input")

# 1. Instantiate pre-trained ImageNet base and lock weights
vgg_base = VGG16(weights='imagenet', include_top=False)
vgg_base.trainable = False

# 2. Feature Extraction
x = vgg_base(image_input)
x = GlobalAveragePooling2D(name="Spatial_Feature_Flattening")(x)

# 3. Interpretation Network (The Diagnostic Head)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x) # Aggressive regularization to prevent overfitting
x = Dense(128, activation='relu')(x)

# Output: 4-Class Probability Distribution
diagnostic_output = Dense(4, activation='softmax', name="Diagnostic_Prediction")(x)

vision_model = Model(inputs=image_input, outputs=diagnostic_output, name="OASIS_Vision_Network")

# 4. Advanced Gradient Optimization
# Using Label Smoothing to reduce overconfidence and mathematical noise
smoothed_loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

vision_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=smoothed_loss,
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

vision_model.summary()

# 4.0 Dynamic Training Protocols & Gradient Optimization

To prevent the custom dense layers from overfitting to the frozen visual base, we enforce strict dynamic callbacks. Early stopping mitigates overfitting, while an adaptive learning rate scheduler resolves loss plateaus.

In [ ]:
# ==============================================================================
# 4.0 TRAINING & VISUALIZATION
# ==============================================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

print("--- INITIATING TRAINING PROTOCOL ---")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# Run the training
history = vision_model.fit(
    train_dataset,
    epochs=25,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)
print("\nSpatial Network Training Complete.")

# Plot the Learning Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o', color='orange')
plt.title('CNN Accuracy', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o', color='red')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o', color='orange')
plt.title('CNN Gradient Loss (Smoothed)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 4.0 TRAINING & VISUALIZATION (WITH CLASS WEIGHTS)
# ==============================================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

print("--- INITIATING TRAINING PROTOCOL (WITH CLASS WEIGHTS) ---")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# Run the training, now including the calculated class weights
history = vision_model.fit(
    train_dataset,
    epochs=25,
    validation_data=val_dataset,
    callbacks=callbacks,
    class_weight=class_weights_dict, # <--- ADDED CLASS WEIGHTS HERE
    verbose=1
)
print("\nSpatial Network Training Complete.")

# Plot the Learning Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o', color='orange')
plt.title('CNN Accuracy', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o', color='red')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o', color='orange')
plt.title('CNN Gradient Loss (Smoothed)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
vision_model.save('/content/drive/MyDrive/Study/MSc AI/data sets/Dementia - Image DL Analysis for 10 min vlog/trained_vgg16_model.keras')
print("Model permanently saved to Drive!")

#5.0 Holistic Clinical Audit (The Skeptic's Review)

Executing a rigorous evaluation on strictly unseen test data. To validate clinical utility, we generate precision/recall metrics and an error matrix to ensure the model successfully differentiates between contiguous stages of cognitive decline.

In [ ]:
# ==============================================================================
# 5.0 CLINICAL AUDIT
# ==============================================================================
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

print("--- EXECUTING CLINICAL AUDIT (AFTER WEIGHTED TRAINING) ---")

# 1. Top-Line Unseen Data Benchmarking
test_loss, test_acc, test_auc = vision_model.evaluate(test_dataset, verbose=0)
print(f"Final Test Accuracy: {test_acc:.4f}")
print(f"Final Test AUC:      {test_auc:.4f}\n")

# 2. Extract Predictions
y_true = []
y_pred_probs = []

for batch in test_dataset:
    inputs, labels = batch
    preds = vision_model.predict(inputs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_probs.extend(preds)

y_true_classes = np.argmax(y_true, axis=1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
class_labels = ['Non-Demented', 'Very Mild', 'Mild', 'Moderate']

# 3. Deep Dive Classification Report
print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_labels))

# 4. Visualizing the Error Matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.title("CNN Spatial Model: True vs Predicted Dementia Stages (Weighted Training)", fontweight='bold')
plt.tight_layout()
plt.show()

### Analyzing Class Imbalance

The `UndefinedMetricWarning` you observed often indicates a class imbalance, especially when some classes have very few samples in the test set. Let's visualize the true label distribution in your `test_dataset` to understand this better.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming y_true_classes and class_labels are available from Enwq6DMfNYV1

# Count the occurrences of each true class
class_counts = pd.Series(y_true_classes).value_counts().sort_index()

# Map numerical indices to class names for better readability
class_distribution = pd.DataFrame({
    'Class': [class_labels[i] for i in class_counts.index],
    'Count': class_counts.values
})

plt.figure(figsize=(10, 6))
sns.barplot(x='Class', y='Count', data=class_distribution, palette='viridis', hue='Class', legend=False)
plt.title('True Class Distribution in Test Dataset', fontweight='bold')
plt.xlabel('Dementia Stage')
plt.ylabel('Number of Samples')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("True Class Distribution:\n", class_distribution)

As you can see from the plot and table above, the 'Moderate' dementia class has a significantly smaller number of samples (likely 5, as indicated by the `support` column in your classification report) compared to the other classes.

This severe imbalance means that:
1.  **Model Difficulty**: The model has very few examples to learn from for the 'Moderate' class, making it hard to predict accurately.
2.  **Metric Calculation**: If the model doesn't predict *any* samples as 'Moderate' (or if all its 'Moderate' predictions are incorrect) and there are also very few true 'Moderate' samples, `sklearn`'s `precision` and `recall` metrics can become ill-defined (e.g., division by zero), leading to the `UndefinedMetricWarning` and values of `0.00` in the classification report for that class.

To address this, you might consider strategies like oversampling the minority class, undersampling the majority classes, or using weighted loss functions during training if you want the model to perform better on these rare classes.

# 6.0 Explainable AI (XAI): Spatial Interpretability via Grad-CAM

In high-stakes clinical diagnostics, a 'Black Box' CNN is insufficient. To satisfy the requirement for model interpretability, we implemened Gradient-weighted Class Activation Mapping (Grad-CAM). This technique extracts the spatial gradients from the final convolutional layer of the VGG16 base, generating a visual heatmap that proves the network is isolating neurologically relevant regions (e.g., ventricular enlargement or cortical shrinkage) rather than learning spurious background noise.

In [ ]:
# ==============================================================================
# 6.0 EXPLAINABLE AI: GRAD-CAM SPATIAL AUDIT
# ==============================================================================

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Assuming vision_model and test_dataset are available from previous execution

def make_gradcam_heatmap_nested(img_array, model, inner_model_name="vgg16", target_layer_name="block5_conv3", pred_index=None):
    inner_model = model.get_layer(inner_model_name)

    vgg_cam_model = tf.keras.Model(
        inner_model.inputs,
        [inner_model.get_layer(target_layer_name).output, inner_model.output]
    )

    classifier_layers = model.layers[2:]

    with tf.GradientTape() as tape:
        last_conv_output, vgg_output = vgg_cam_model(img_array)
        tape.watch(last_conv_output)

        x = vgg_output
        for layer in classifier_layers:
            x = layer(x)
        preds = x

        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# 1. Select a sample image from the test dataset safely
for images, labels in test_dataset.take(1):
    sample_image = images[0:1] # Keep batch dimension (1, 224, 224, 3)
    true_label_idx = np.argmax(labels[0])
    break

# 2. Generate Heatmap using the nested workaround
print("Tracing gradients through VGG16 block5_conv3...")
heatmap = make_gradcam_heatmap_nested(sample_image, vision_model)

# 3. Superimpose the heatmap on the original image
img = sample_image[0].numpy()
img = (img - np.min(img)) / (np.max(img) - np.min(img)) # Normalize for display

heatmap = np.uint8(255 * heatmap)
jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap]
jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap) / 255.0

superimposed_img = jet_heatmap * 0.4 + img
superimposed_img = np.clip(superimposed_img, 0, 1)

class_names = ['Non-Demented', 'Very Mild', 'Mild', 'Moderate']

# 4. Plot the Gestalt Visualisation
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(f"Original MRI (True: {class_names[true_label_idx]})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(superimposed_img)
plt.title("Grad-CAM Spatial Audit")
plt.axis('off')

plt.tight_layout()
plt.show()

# 7.0 Prescriptive Analytics: Reinforcement Learning Proof-of-Concept

Mapping the diagnostic stage predicted by our CNN to a Markov Decision Process (MDP). A Q-Learning agent uses the Bellman Equation to learn the mathematically optimal clinical intervention, balancing treatment efficacy against the risks of over-medication.

In [ ]:
import random
import numpy as np
import pandas as pd # Import pandas for better display
import sys # Import sys for flushing output
from tqdm.notebook import tqdm # Import tqdm for progress bar

print("--- INITIALIZING Q-LEARNING CLINICAL AGENT ---")
sys.stdout.flush()

# States map to our 4 CNN Output Classes
states = [0, 1, 2, 3]
state_names = ['Non-Demented', 'Very Mild Dementia', 'Mild Dementia', 'Moderate Dementia']

actions = [0, 1, 2, 3]
action_names = ['Routine Checkup', 'Cognitive Therapy', 'Mild Medication', 'Aggressive Treatment']

q_table = np.zeros((len(states), len(actions)))

learning_rate = 0.1
discount_factor = 0.9
exploration_rate = 1.0 # Start with higher exploration
max_exploration_rate = 1.0
min_exploration_rate = 0.01
exploration_decay_rate = 0.001 # Rate at which exploration decreases
epochs = 5000

# The Reward Matrix
reward_matrix = np.array([
    [ 10,  -5, -10, -20],
    [  0,  10,  -5, -10],
    [ -5,   5,  10,   0],
    [-20,  -5,   5,  10]
])

# Wrap the epoch loop with tqdm for a progress bar
for epoch in tqdm(range(epochs), desc="Training RL Agent"):
    state = random.choice(states)

    # Exploration-exploitation trade-off
    if random.uniform(0, 1) < exploration_rate:
        action = random.choice(actions)
    else:
        action = np.argmax(q_table[state])

    reward = reward_matrix[state, action]

    # Simplified next state logic (can be made more complex if needed)
    next_state = max(0, state - 1) if action >= state else state

    best_next_action = np.max(q_table[next_state])
    q_table[state, action] = q_table[state, action] + learning_rate * (reward + discount_factor * best_next_action - q_table[state, action])

    # Decay exploration rate
    exploration_rate = min_exploration_rate + \
                       (max_exploration_rate - min_exploration_rate) * np.exp(-exploration_decay_rate*epoch)

print("RL Agent Training Complete. Extracting Optimal Clinical Policy...\n")
sys.stdout.flush()

# Display the Q-table
q_table_df = pd.DataFrame(q_table, index=state_names, columns=action_names)
print("--- FINAL Q-TABLE ---")
print(q_table_df)
sys.stdout.flush()

print("\n--- PRESCRIBED CLINICAL ACTIONS BASED ON CNN DIAGNOSIS ---")
policy_data = []
for s in states:
    best_action_idx = np.argmax(q_table[s])
    policy_data.append(
        {
            "CNN Predicted Stage": state_names[s],
            "RL Prescribes Action": action_names[best_action_idx]
        }
    )
policy_df = pd.DataFrame(policy_data)
print(policy_df)
sys.stdout.flush()

#8.0 NLP Layer: Automated Clinical Reporting Summary

In [ ]:
# =========================================================================
# 8.0 NLP LAYER - FOR NON-GENERATIVE SUMMARISATION
# =========================================================================

# Step 1: Define class labels to match your CNN output order
class_names = ['Non-Demented', 'Very Mild Dementia', 'Mild Dementia', 'Moderate Dementia']

# Step 2: Create a helper to convert one-hot labels to class index
def one_hot_to_index(label_vector):
    return int(np.argmax(label_vector))

# Step 3: Create a helper to turn probabilities into readable percentages
def format_probabilities(prob_vector):
    return {
        class_names[i]: round(float(prob_vector[i]) * 100, 2)
        for i in range(len(class_names))
    }

# Step 4: Create a simple rule-based explanation for confidence
def confidence_band(confidence_score):
    if confidence_score >= 0.85:
        return "high confidence"
    elif confidence_score >= 0.60:
        return "moderate confidence"
    else:
        return "low confidence"

# Step 5: Map RL action names to a short clinical interpretation
def action_explanation(action_name):
    explanations = {
        'Routine Checkup': 'The suggested next step is routine monitoring rather than immediate intervention.',
        'Cognitive Therapy': 'The suggested next step is supportive cognitive intervention at an early stage.',
        'Mild Medication': 'The suggested next step is a moderate treatment response based on predicted progression.',
        'Aggressive Treatment': 'The suggested next step is urgent and intensive clinical escalation.'
    }
    return explanations.get(action_name, 'No explanation available.')

# Step 6: Build a structured non-generative report
def build_case_report(predicted_class, confidence, probability_dict, prescribed_action):
    top_probs = sorted(probability_dict.items(), key=lambda x: x[1], reverse=True)

    report = f"""
    Diagnostic Summary
    ------------------
    Predicted class: {predicted_class}
    Confidence level: {confidence_band(confidence)}
    Top class probabilities:
    1. {top_probs[0][0]}: {top_probs[0][1]}%
    2. {top_probs[1][0]}: {top_probs[1][1]}%

    Prescriptive recommendation:
    {prescribed_action}

    Action interpretation:
    {action_explanation(prescribed_action)}

    Final summary:
    The MRI scan was classified as {predicted_class} with {confidence_band(confidence)}.
    Based on the reinforcement learning policy, the recommended action is {prescribed_action}.
    """
    return report.strip()

# Step 7: Run inference on a small batch from your test dataset
for image_batch, label_batch in test_dataset.take(1):
    prediction_batch = vision_model.predict(image_batch, verbose=0)

    for i in range(min(3, len(prediction_batch))):  # analyse first 3 examples
        true_idx = one_hot_to_index(label_batch[i].numpy())
        pred_idx = int(np.argmax(prediction_batch[i]))
        predicted_class = class_names[pred_idx]
        confidence = float(np.max(prediction_batch[i]))
        probability_dict = format_probabilities(prediction_batch[i])

        # Use your trained RL policy
        best_action_idx = int(np.argmax(q_table[pred_idx]))
        prescribed_action = action_names[best_action_idx]

        report = build_case_report(
            predicted_class=predicted_class,
            confidence=confidence,
            probability_dict=probability_dict,
            prescribed_action=prescribed_action
        )

        print(f"\n===== CASE {i+1} =====")
        print(f"True label: {class_names[true_idx]}")
        print(report)

# Step 8: Store outputs in a dataframe

results = []

for image_batch, label_batch in test_dataset.take(1):
    prediction_batch = vision_model.predict(image_batch, verbose=0)

    for i in range(min(5, len(prediction_batch))):
        true_idx = int(np.argmax(label_batch[i].numpy()))
        pred_idx = int(np.argmax(prediction_batch[i]))
        confidence = float(np.max(prediction_batch[i]))
        best_action_idx = int(np.argmax(q_table[pred_idx]))

        results.append({
            "true_label": class_names[true_idx],
            "predicted_label": class_names[pred_idx],
            "confidence": round(confidence, 4),
            "recommended_action": action_names[best_action_idx]
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:
import glob
import subprocess

# Look everywhere for the notebook, including inside folders like sample_data
notebooks = glob.glob('**/*.ipynb', recursive=True)

# Ignore hidden system files just in case
notebooks = [f for f in notebooks if '.ipynb_checkpoints' not in f]

if not notebooks:
    print("❌ Error: No .ipynb file found anywhere.")
else:
    target_file = notebooks[0]
    print(f"⏳ Found your file hiding at: '{target_file}'. Converting to HTML now...")

    # Run the conversion command
    subprocess.run(['jupyter', 'nbconvert', '--to', 'html', target_file])
    print("✅ Conversion complete! Now, go click the Refresh icon at the top of the files menu.")